# CreditWise — Preprocessing Pipeline

**Notebook 02 of 04**

This notebook documents and validates the preprocessing pipeline:
- Data cleaning (EDA-driven)
- Feature engineering (7 justified features)
- Train/test splitting
- Imputation and scaling
- Verification that no data leakage occurs

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.data_loader import load_raw_data, clean_raw_data
from src.feature_engineering import engineer_features, get_all_feature_names, ENGINEERED_FEATURE_NAMES
from src.preprocessing import build_preprocessing_pipeline, split_data, fit_pipeline
from src.config import EXPECTED_FEATURE_COLUMNS, TARGET_COLUMN

plt.rcParams.update({'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
                     'axes.labelcolor':'#e2e8f0','xtick.color':'#94a3b8',
                     'ytick.color':'#94a3b8','text.color':'#e2e8f0','figure.dpi':110})
print('Ready.')

## 1. Load and Clean

In [ ]:
df_raw = load_raw_data()
df = clean_raw_data(df_raw)
print(f'Raw rows : {len(df_raw):,}')
print(f'Clean rows: {len(df):,}  (dropped {len(df_raw)-len(df)} duplicates, replaced 96/98 codes with NaN)')

## 2. Feature Engineering

Seven financially justified features are added:

| Feature | Justification |
|---|---|
| log_revolving_utilization | Log-compresses extreme right-skew |
| log_debt_ratio | Same; 16.3% of values > 100 |
| log_monthly_income | Income is log-normally distributed |
| total_past_due | Cumulative delinquency signal |
| has_past_due | Binary: any delinquency |
| income_per_dependent | Affordability per household member |
| credit_line_density | Credit lines / effective credit age |

In [ ]:
df_eng = engineer_features(df)
all_features = get_all_feature_names()

print(f'Original features : {len(EXPECTED_FEATURE_COLUMNS)}')
print(f'Engineered features: {len(ENGINEERED_FEATURE_NAMES)}')
print(f'Total features    : {len(all_features)}')
print('\nEngineered features added:')
for f in ENGINEERED_FEATURE_NAMES:
    print(f'  + {f}')
print()
display(df_eng[ENGINEERED_FEATURE_NAMES].describe().round(3))

In [ ]:
# Visualise engineered features
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(ENGINEERED_FEATURE_NAMES):
    data = df_eng[col].dropna()
    axes[i].hist(data.clip(data.quantile(0.01), data.quantile(0.99)), bins=60,
                 color='#818cf8', edgecolor='#0f172a', alpha=0.85)
    axes[i].set_title(col, fontsize=9)
    axes[i].grid(axis='y', alpha=0.4)
axes[-1].set_visible(False)
plt.suptitle('Engineered Feature Distributions', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Train / Test Split

In [ ]:
X = df_eng[all_features]
y = df_eng[TARGET_COLUMN]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} rows  |  Default rate: {y_train.mean()*100:.2f}%')
print(f'Test : {len(X_test):,}  rows  |  Default rate: {y_test.mean()*100:.2f}%')
print()
print('Stratification check — class distribution preserved:')
for split_name, y_s in [('Original', y), ('Train', y_train), ('Test', y_test)]:
    print(f'  {split_name:10s}: class0={y_s.value_counts()[0]:,} ({(1-y_s.mean())*100:.1f}%)'  
          f'  class1={y_s.value_counts()[1]:,} ({y_s.mean()*100:.1f}%)')

## 4. Preprocessing Pipeline (Imputation + Scaling)

In [ ]:
pipeline = build_preprocessing_pipeline(scale_features=True)
print('Pipeline steps:', [step[0] for step in pipeline.steps])

X_train_tf, X_test_tf, fitted_pipeline = fit_pipeline(pipeline, X_train, X_test)

print(f'\nX_train shape before: {X_train.shape}  →  after: {X_train_tf.shape}')
print(f'X_test shape  before: {X_test.shape}   →  after: {X_test_tf.shape}')
print(f'\nNaN in X_train_tf: {np.isnan(X_train_tf).sum()}')
print(f'NaN in X_test_tf : {np.isnan(X_test_tf).sum()}')

In [ ]:
# Verify imputed medians come from training set only
imputer = fitted_pipeline.named_steps['imputer']
imputed_medians = dict(zip(all_features, imputer.statistics_))
print('Imputed medians (from TRAINING data only):')
# Only show features that had missing values
for feat in ['MonthlyIncome', 'NumberOfDependents']:
    idx = all_features.index(feat)
    print(f'  {feat}: {imputer.statistics_[idx]:.2f}')

## 5. Data Leakage Check

Critical: the preprocessing pipeline is fit ONLY on training data. The test set's statistics are never used to impute or scale training data.

In [ ]:
# Compare imputed median vs true test-set median — should be slightly different (no leakage)
income_idx = all_features.index('MonthlyIncome')
train_median = imputer.statistics_[income_idx]
test_median_true = X_test['MonthlyIncome'].median()

print('Leakage check for MonthlyIncome:')
print(f'  Imputer median (from training data) : {train_median:.2f}')
print(f'  True test-set median                : {test_median_true:.2f}')
print(f'  Difference                          : {abs(train_median-test_median_true):.2f}')
print()
print('Small difference expected — confirms NO data leakage.')

## 6. Summary

The preprocessing pipeline is ready:

```
Raw CSV → clean_raw_data() → engineer_features() → train_test_split() →
    Pipeline.fit_transform(X_train) → X_train_tf
    Pipeline.transform(X_test)      → X_test_tf   [NO fit on test]
```

The fitted pipeline is saved to `models/preprocessing_pipeline.joblib`
and loaded identically at inference time — preventing training/inference mismatch.